# Clase 3 · Laboratorio — ETL completo: Saber 11 → Bodega en SQLite

**Trabajo en parejas · 90 min.**

En la pre-clase construyeron `dim_colegio` y un hecho parcial. Hoy completan la bodega:
- Agregar `dim_tiempo` y `dim_geografia`
- Actualizar `hecho_resultados` con las 3 claves de referencia
- Cargar el modelo estrella completo a SQLite
- Ejecutar 3 consultas analíticas reales

**Checkpoint del profesor a los 50 min** (después de la Tarea 3).

In [1]:
import pandas as pd
import sqlite3

CSV = "saber11_muestra_500k.csv"
DB  = "saber11_lab_etl.db"

df_raw = pd.read_csv(CSV, dtype=str)
df = df_raw.copy()

# Transformaciones base
cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
for col in cols_puntaje:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE",
            "COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"]:
    df[col] = df[col].str.strip().str.upper()

df["PERIODO"] = pd.to_numeric(df["PERIODO"], errors="coerce").astype("Int64")

print(f"Datos listos: {len(df):,} filas x {df.shape[1]} columnas")

Datos listos: 500,000 filas x 22 columnas


## Tarea 1 (10 min) — Reconstruir dim_colegio + añadir dim_tiempo y dim_geografia

Ya construyeron `dim_colegio` en la pre-clase. Aquí la reconstruyen rápido y añaden las dos dimensiones que faltan.

| Dimensión | Columnas (además del ID) |
|---|---|
| `dim_colegio` | COLE_NATURALEZA, COLE_JORNADA, COLE_CALENDARIO, COLE_BILINGUE |
| `dim_tiempo` | PERIODO |
| `dim_geografia` | COLE_DEPTO_UBICACION, COLE_MCPIO_UBICACION |

In [2]:
# dim_colegio (igual que en pre-clase)
dim_colegio = (
    df[["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"]]
    .drop_duplicates().reset_index(drop=True)
)
dim_colegio.insert(0, "colegio_id", dim_colegio.index + 1)
assert dim_colegio["colegio_id"].is_unique

# dim_tiempo -- parto PERIODO en anio y quarter
# el PERIODO viene como 20194 -> anio=2019, quarter=4
dim_tiempo = df[["PERIODO"]].drop_duplicates().reset_index(drop=True)
dim_tiempo.insert(0, "tiempo_id", dim_tiempo.index + 1)
dim_tiempo["anio"] = dim_tiempo["PERIODO"] // 10
dim_tiempo["quarter"] = dim_tiempo["PERIODO"] % 10
assert dim_tiempo["tiempo_id"].is_unique

# dim_geografia con geo_id (COLE_DEPTO_UBICACION + COLE_MCPIO_UBICACION)
dim_geografia = (
    df[["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"]]
    .drop_duplicates().reset_index(drop=True)
)
dim_geografia.insert(0, "geo_id", dim_geografia.index + 1)
assert dim_geografia["geo_id"].is_unique

print(f"dim_colegio: {len(dim_colegio)} | dim_tiempo: {len(dim_tiempo)} | dim_geografia: {len(dim_geografia)}")
print("\ndim_tiempo:")
print(dim_tiempo)

dim_colegio: 40 | dim_tiempo: 4 | dim_geografia: 10

dim_tiempo:
   tiempo_id  PERIODO  anio  quarter
0          1    20194  2019        4
1          2    20224  2022        4
2          3    20214  2021        4
3          4    20204  2020        4


## Tarea 2 (20 min) — hecho_resultados con las 3 claves de referencia

En la pre-clase el hecho solo tenía `colegio_id`. Ahora hay que agregar `tiempo_id` y `geo_id`.

1. Une `df` con las 3 dimensiones mediante `.merge(..., how='left')`.
2. El hecho final debe tener: `colegio_id`, `tiempo_id`, `geo_id` + los 6 puntajes.
3. **Validación:** `len(hecho_resultados) == len(df)` — si falla, hay un join mal configurado.

In [3]:
# Unir con las 3 dimensiones para obtener las claves de referencia
df_h = df.merge(dim_colegio,
                on=["COLE_NATURALEZA","COLE_JORNADA","COLE_CALENDARIO","COLE_BILINGUE"],
                how="left")

# uno con dim_tiempo por PERIODO
df_h = df_h.merge(dim_tiempo[["tiempo_id", "PERIODO"]], on="PERIODO", how="left")

# uno con dim_geografia por depto+municipio
df_h = df_h.merge(dim_geografia, on=["COLE_DEPTO_UBICACION","COLE_MCPIO_UBICACION"], how="left")

cols_puntaje = ["PUNT_LECTURA_CRITICA","PUNT_MATEMATICAS","PUNT_C_NATURALES",
                "PUNT_SOCIALES_CIUDADANAS","PUNT_INGLES","PUNT_GLOBAL"]
hecho_resultados = df_h[["colegio_id","tiempo_id","geo_id"] + cols_puntaje].copy()

assert len(hecho_resultados) == len(df), f"Perdimos {len(df)-len(hecho_resultados)} filas"
print(f"hecho_resultados: {len(hecho_resultados):,} filas con 3 FKs OK")
hecho_resultados.head(3)

hecho_resultados: 500,000 filas con 3 FKs OK


,colegio_id,tiempo_id,geo_id,PUNT_LECTURA_CRITICA,PUNT_MATEMATICAS,PUNT_C_NATURALES,PUNT_SOCIALES_CIUDADANAS,PUNT_INGLES,PUNT_GLOBAL
0,1,1,1,48,51,26,14,67,194
1,1,2,2,77,32,64,48,48,199
2,2,3,3,65,43,63,84,48,268


## Tarea 3 (15 min) — Cargar el modelo estrella completo a SQLite

Carga las 4 tablas a la BD. Luego verifica contando las filas de cada tabla.

In [4]:
conn = sqlite3.connect(DB)

# cargo las 4 tablas con to_sql (si ya existian las reemplazo)
dim_colegio.to_sql("dim_colegio", conn, if_exists="replace", index=False)
dim_tiempo.to_sql("dim_tiempo", conn, if_exists="replace", index=False)
dim_geografia.to_sql("dim_geografia", conn, if_exists="replace", index=False)
hecho_resultados.to_sql("hecho_resultados", conn, if_exists="replace", index=False)

# Verificar
for tabla in ["dim_colegio", "dim_tiempo", "dim_geografia", "hecho_resultados"]:
    n = conn.execute(f"SELECT COUNT(*) FROM {tabla}").fetchone()[0]
    print(f"  {tabla}: {n:,} filas")

  dim_colegio: 40 filas
  dim_tiempo: 4 filas
  dim_geografia: 10 filas
  hecho_resultados: 500,000 filas


## ⏸ Checkpoint del profesor (10 min)

Revisamos juntos:
- ¿Alguien perdió filas en la Tarea 2? Diagnóstico rápido.
- ¿Por qué las bodegas de datos NO imponen restricciones de FK en la base de datos?
- ¿Qué representa una fila del hecho_resultados? ¿Un estudiante, un colegio, un período?

**Mis respuestas rápidas:**
- No perdí filas (ya lo validé con el `assert` de la Tarea 2), pero si me hubiera pasado, lo primero que revisaría es si alguna de las columnas usadas para el join tiene espacios de más o mayúsculas/minúsculas distintas entre `df` y la dimensión (por eso hicimos `.str.strip().str.upper()` al principio).
- Las bodegas no imponen FK en la base de datos porque el ETL ya garantizó la integridad antes de cargar los datos (por eso hicimos los `assert` en pandas). Además, no tener esas restricciones hace que las cargas masivas sean mucho más rápidas, que es justo lo que se necesita en una bodega con millones de filas.
- Una fila de `hecho_resultados` representa un estudiante en un período específico (no un colegio ni un departamento), porque esa es la granularidad más fina que tenemos: cada estudiante presentó el examen una sola vez por período.

## Tarea 4 (30 min) — Diagrama del modelo estrella + Consultas SQL

### Parte A — Diagrama del modelo (5 min)

Aquí está el modelo que construyeron. Añadan en el Markdown de abajo los atributos de cada tabla:

```
                dim_tiempo
         (tiempo_id, PERIODO, anio, quarter)
                     |
dim_colegio  <--  hecho_resultados  -->  dim_geografia
(colegio_id,      (colegio_id (ref.),      (geo_id,
NATURALEZA,        tiempo_id (ref.),        DEPTO_UBICACION,
JORNADA,           geo_id (ref.),           MCPIO_UBICACION)
CALENDARIO,        PUNT_LECTURA_CRITICA,
BILINGUE)          PUNT_MATEMATICAS,
                   PUNT_C_NATURALES,
                   PUNT_SOCIALES_CIUDADANAS,
                   PUNT_INGLES,
                   PUNT_GLOBAL)
```

In [5]:
# Consulta 1: promedio de PUNT_GLOBAL por naturaleza del colegio (oficial vs no oficial)
q1 = """
SELECT c.COLE_NATURALEZA,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_colegio c ON h.colegio_id = c.colegio_id
GROUP BY c.COLE_NATURALEZA
ORDER BY prom_global DESC
"""
r1 = pd.read_sql(q1, conn)
print("Consulta 1:")
print(r1)

Consulta 1:
  COLE_NATURALEZA  prom_global  n_estudiantes
0      NO OFICIAL        248.8         140091
1         OFICIAL        248.6         359909


In [6]:
# Consulta 2: top 5 departamentos con mayor promedio de PUNT_MATEMATICAS
q2 = """
SELECT g.COLE_DEPTO_UBICACION,
       ROUND(AVG(h.PUNT_MATEMATICAS), 1) AS prom_matematicas,
       COUNT(*) AS n_estudiantes
FROM hecho_resultados h
JOIN dim_geografia g ON h.geo_id = g.geo_id
GROUP BY g.COLE_DEPTO_UBICACION
ORDER BY prom_matematicas DESC
LIMIT 5
"""
r2 = pd.read_sql(q2, conn)
print("Consulta 2:")
print(r2)

Consulta 2:
  COLE_DEPTO_UBICACION  prom_matematicas  n_estudiantes
0            SANTANDER              48.4          49747
1          BOGOTÁ D.C.              48.4          50115
2            ATLÁNTICO              48.4          49692
3      VALLE DEL CAUCA              48.3          50143
4               NARIÑO              48.3          50088


In [7]:
# Consulta 3: por año y quarter, en que periodo mejoro mas el puntaje
q3 = """
SELECT t.anio, t.quarter,
       ROUND(AVG(h.PUNT_GLOBAL), 1) AS prom_global
FROM hecho_resultados h
JOIN dim_tiempo t ON h.tiempo_id = t.tiempo_id
GROUP BY t.anio, t.quarter
ORDER BY t.anio ASC, t.quarter ASC
"""
r3 = pd.read_sql(q3, conn)
print("Consulta 3:")
print(r3)

conn.close()

Consulta 3:
   anio  quarter  prom_global
0  2019        4        248.9
1  2020        4        248.6
2  2021        4        248.7
3  2022        4        248.4


### Conclusión

Mirando los resultados de las 3 consultas, escribe 3-4 frases:
- ¿Qué brecha de puntaje hay entre colegios oficiales y no oficiales?
- ¿La jornada con mejor puntaje coincide con lo que esperabas?

_(Mis 3-4 frases):_ La brecha entre oficiales y no oficiales es minima: en la Consulta 1 los no oficiales quedan apenas por encima (0.2 puntos de diferencia sobre 500: 248.8 contra 248.6), asi que en este dataset simulado casi no importa si el colegio es oficial o no. En la Consulta 2 los 5 departamentos con mejor promedio en matematicas estan todos muy parejos entre si (la diferencia entre el primero y el quinto es de menos de 1 punto), lo que sugiere que en esta muestra el departamento tampoco influye demasiado en matematicas. Con la Consulta 3 se ve que los 4 periodos (2019 a 2022) tienen un promedio de PUNT_GLOBAL casi identico, sin una tendencia clara de mejora o caida a lo largo de los años. La verdad esperaba ver diferencias mas marcadas por naturaleza del colegio (pense que los no oficiales sacarian bastante mas), pero parece que este dataset esta armado para que esas variables no tengan mucho poder explicativo por si solas.

---

## Reflexión final — ¿Qué dimensión faltó?

Revisa las columnas del CSV de Saber 11 que **no usaste** en ninguna dimensión:

```
ESTU_GENERO, ESTU_FECHANACIMIENTO
FAMI_ESTRATOVIVIENDA, FAMI_TIENEINTERNET
FAMI_EDUCACIONMADRE, FAMI_EDUCACIONPADRE
DESEMP_INGLES
```

**Preguntas:**
1. ¿A qué dimensión pertenecen estas columnas? ¿Cómo la llamarías?
2. ¿Cuáles columnas incluirías en esa dimensión y cuáles dejarías fuera? ¿Por qué?
3. Bosqueja el código para construirla (solo la estructura — no es necesario ejecutarla):

```python
# dim_??? = (
#     df[["...", "...", "..."]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# dim_???.insert(0, "???_id", dim_???.index + 1)
```

_Tus respuestas:_

1. Estas columnas hablan del estudiante y su familia, no del colegio ni del lugar, asi que yo la llamaria `dim_estudiante` (o `dim_perfil_estudiante`).

2. Incluiria ESTU_GENERO, FAMI_ESTRATOVIVIENDA, FAMI_TIENEINTERNET, FAMI_EDUCACIONMADRE y FAMI_EDUCACIONPADRE porque son datos categoricos que se repiten mucho entre estudiantes (pocos valores distintos), que es justo el tipo de columna que conviene sacar a una dimension aparte. Dejaria fuera ESTU_FECHANACIMIENTO porque es casi unica por estudiante (no se repite casi nunca) y DESEMP_INGLES porque en realidad es mas una medida/resultado del examen que un atributo descriptivo del estudiante, se parece mas a algo que iria en la tabla de hechos.

3. Boceto del codigo:

```python
# dim_estudiante = (
#     df[["ESTU_GENERO", "FAMI_ESTRATOVIVIENDA", "FAMI_TIENEINTERNET",
#         "FAMI_EDUCACIONMADRE", "FAMI_EDUCACIONPADRE"]]
#     .drop_duplicates()
#     .reset_index(drop=True)
# )
# dim_estudiante.insert(0, "estudiante_perfil_id", dim_estudiante.index + 1)
```

## Entrega

- Suban este notebook completado a Moodle antes de las 23:59.
- Nombre: `apellido1_apellido2_clase03_lab.ipynb`.